# 🌀 Dark Manifold v3 — Hamiltonian Dynamics + Delta Loss

## The Problem with v2

Previous models learned to predict **flat lines** because:
- Standard MSE averages over all metabolites
- 90% of metabolites are nearly constant
- Model learns: "predict constant = low loss"

## The Fix: Dynamic Loss

| Component | Weight | What it does |
|-----------|--------|-------------|
| **state_loss** | 1x | Standard MSE |
| **delta_loss** | 10x | Penalize wrong CHANGE |
| **direction_loss** | 5x | At least get UP/DOWN right |
| **energy_loss** | 1x | Hamiltonian conservation |

**Result:** Model MUST learn dynamics, not just mean.

In [ ]:
#@title 1️⃣ Upload Data
from google.colab import files
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import re
from tqdm import tqdm

print("Upload syn3a_real_data.pkl:")
uploaded = files.upload()

with open('syn3a_real_data.pkl', 'rb') as f:
    raw_data = pickle.load(f)

print("Loaded!")

In [ ]:
#@title 2️⃣ Parse Real Data

dfs = raw_data['all_dataframes']
rxns_df = pd.DataFrame(dfs['reconstruction.xlsx:Reactions'])
mets_df = pd.DataFrame(dfs['reconstruction.xlsx:Metabolites'])

# Build metabolite index
metabolites = mets_df['Metabolite ID'].tolist()
met_to_idx = {m: i for i, m in enumerate(metabolites)}

# Get genes from GPR rules
genes = set()
gene_to_rxns = {}
gpr_rules = rxns_df['GPR rule'].tolist()

for j, gpr in enumerate(gpr_rules):
    if pd.isna(gpr):
        continue
    gene_ids = re.findall(r'MMSYN1_\d+', str(gpr))
    for g in gene_ids:
        genes.add(g)
        if g not in gene_to_rxns:
            gene_to_rxns[g] = []
        gene_to_rxns[g].append(j)

genes = sorted(list(genes))
gene_to_idx = {g: i for i, g in enumerate(genes)}

n_genes = len(genes)
n_mets = len(metabolites)
n_rxns = len(rxns_df)

# Build stoichiometry
S = np.zeros((n_mets, n_rxns))
for j, row in rxns_df.iterrows():
    equation = row['Reaction equation']
    for i, met in enumerate(mets_df['Metabolite name'].tolist()):
        met_id = mets_df.iloc[i]['Metabolite ID']
        if met_id in met_to_idx and met in equation:
            idx = met_to_idx[met_id]
            left_part = equation.split('-->')[0] if '-->' in equation else equation.split('<==>')[0]
            S[idx, j] = -1 if met in left_part else +1

# Gene-reaction matrix
G = np.zeros((n_genes, n_rxns))
for g, rxn_indices in gene_to_rxns.items():
    g_idx = gene_to_idx[g]
    for r_idx in rxn_indices:
        G[g_idx, r_idx] = 1

print(f"Genes: {n_genes}, Metabolites: {n_mets}, Reactions: {n_rxns}")
print(f"Stoichiometry: {S.shape}, non-zero: {np.count_nonzero(S)}")

In [ ]:
#@title 3️⃣ Dynamic Loss Function (THE KEY FIX)

class HamiltonianDynamicsLoss(nn.Module):
    """
    Loss that forces learning DYNAMICS, not just mean values.
    
    Standard MSE: model learns to predict constant (mean)
    Dynamic loss: model MUST predict the CHANGE correctly
    """
    
    def __init__(
        self,
        delta_weight: float = 10.0,
        direction_weight: float = 5.0,
        energy_weight: float = 1.0,
    ):
        super().__init__()
        self.delta_weight = delta_weight
        self.direction_weight = direction_weight
        self.energy_weight = energy_weight
    
    def forward(self, pred_next, true_next, true_curr, energies=None):
        """
        Compute combined dynamic loss.
        
        Args:
            pred_next: (B, D) predicted next state
            true_next: (B, D) true next state
            true_curr: (B, D) current state
            energies: optional (2,) or (2, B) for energy conservation
        """
        # 1. State loss (standard MSE)
        state_loss = F.mse_loss(pred_next, true_next)
        
        # 2. Delta loss (THE KEY) - penalize wrong CHANGE
        pred_delta = pred_next - true_curr
        true_delta = true_next - true_curr
        delta_loss = F.mse_loss(pred_delta, true_delta)
        
        # 3. Direction loss - at least get sign right
        # Use soft sign for differentiability
        pred_sign = torch.tanh(pred_delta * 10)
        true_sign = torch.tanh(true_delta * 10)
        direction_loss = F.mse_loss(pred_sign, true_sign)
        
        # 4. Energy conservation (if provided)
        energy_loss = torch.tensor(0.0, device=pred_next.device)
        if energies is not None:
            # Energy should be approximately constant
            energy_var = energies.var()
            energy_loss = energy_var / (energies.mean().abs() + 1e-6)
        
        # Total
        total = (
            state_loss
            + self.delta_weight * delta_loss
            + self.direction_weight * direction_loss
            + self.energy_weight * energy_loss
        )
        
        metrics = {
            'state': state_loss.item(),
            'delta': delta_loss.item(),
            'direction': direction_loss.item(),
            'energy': energy_loss.item() if isinstance(energy_loss, torch.Tensor) else energy_loss,
        }
        
        return total, metrics

print("✅ HamiltonianDynamicsLoss defined")

In [ ]:
#@title 4️⃣ Model with Hamiltonian Dynamics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class HamiltonianCellDynamics(nn.Module):
    """Hamiltonian dynamics: dq/dt = ∂H/∂p, dp/dt = -∂H/∂q"""
    
    def __init__(self, state_dim, hidden_dim=256):
        super().__init__()
        self.state_dim = state_dim
        
        # Potential energy V(q)
        self.potential_net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )
        
        # Mass matrix
        self.log_mass = nn.Parameter(torch.zeros(state_dim))
    
    @property
    def mass_inv(self):
        return torch.exp(-self.log_mass).clamp(0.1, 10.0)
    
    def hamiltonian(self, q, p):
        T = 0.5 * (p ** 2 * self.mass_inv).sum(dim=-1)
        V = self.potential_net(q).squeeze(-1)
        return T + V
    
    def step(self, q, p, dt=0.01):
        """Symplectic Euler - works in both train and eval mode"""
        with torch.enable_grad():
            q_input = q.detach().clone().requires_grad_(True)
            V = self.potential_net(q_input).sum()
            dV_dq = torch.autograd.grad(V, q_input, create_graph=self.training)[0]
        
        p_new = p - dt * dV_dq
        q_new = q + dt * p_new * self.mass_inv
        
        return q_new, p_new


class DarkManifoldV3(nn.Module):
    """
    Dark Manifold v3 with Hamiltonian dynamics.
    
    Differences from v2:
    - Evolves via Hamiltonian (energy conserving)
    - Genes modulate potential energy landscape
    - Momentum initialized from gene-metabolite interaction
    """
    
    def __init__(self, n_genes, n_metabolites, n_reactions, S, G, hidden_dim=256):
        super().__init__()
        
        self.n_genes = n_genes
        self.n_metabolites = n_metabolites
        self.n_reactions = n_reactions
        
        self.register_buffer('S', torch.tensor(S, dtype=torch.float32))
        self.register_buffer('G', torch.tensor(G, dtype=torch.float32))
        
        # Gene encoder
        self.gene_encoder = nn.Sequential(
            nn.Linear(n_genes, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        
        # Hamiltonian dynamics
        self.hamiltonian = HamiltonianCellDynamics(n_metabolites, hidden_dim)
        
        # Momentum generator (genes → initial momentum)
        self.momentum_net = nn.Sequential(
            nn.Linear(hidden_dim + n_metabolites, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, n_metabolites),
            nn.Tanh(),
        )
        
        # Gene regulation
        self.W_reg = nn.Parameter(torch.randn(n_genes, n_genes) * 0.01)
    
    def forward(self, gene_expr, met_conc, dt=0.01, n_steps=1):
        B = gene_expr.shape[0]
        
        # Gene regulation
        reg = torch.tanh(gene_expr @ self.W_reg.T) * 0.1
        regulated = gene_expr + reg
        
        # Encode genes
        gene_hidden = self.gene_encoder(regulated)
        
        # Generate initial momentum
        p0 = self.momentum_net(torch.cat([gene_hidden, met_conc], dim=-1)) * 0.1
        
        # Evolve
        q, p = met_conc, p0
        energies = [self.hamiltonian.hamiltonian(q, p)]
        
        for _ in range(n_steps):
            q, p = self.hamiltonian.step(q, p, dt)
            q = q.clamp(min=0)
            energies.append(self.hamiltonian.hamiltonian(q, p))
        
        return {
            'next_metabolites': q,
            'momentum': p,
            'energies': torch.stack(energies),
            'initial_momentum': p0,
        }


# Create model
model = DarkManifoldV3(
    n_genes=n_genes,
    n_metabolites=n_mets,
    n_reactions=n_rxns,
    S=S,
    G=G,
    hidden_dim=256,
).to(device)

print(f"\nDarkManifoldV3 created:")
print(f"  Genes: {model.n_genes}")
print(f"  Metabolites: {model.n_metabolites}")
print(f"  Reactions: {model.n_reactions}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
#@title 5️⃣ Trajectory Generator (with clear dynamics)

class DynamicTrajectoryGenerator:
    """Generate trajectories with CLEAR dynamics (not flat!)"""
    
    def __init__(self, S, G, n_genes, n_mets, n_rxns):
        self.S = torch.tensor(S, dtype=torch.float32)
        self.G = torch.tensor(G, dtype=torch.float32)
        self.n_genes = n_genes
        self.n_mets = n_mets
    
    def simulate(self, n_steps=50, dt=0.01, batch_size=1):
        gene_expr = torch.rand(batch_size, self.n_genes) * 2.0
        met_conc = torch.rand(batch_size, self.n_mets) * 2.0 + 0.1
        
        # Create TRENDS (some up, some down)
        trends = torch.randn(batch_size, self.n_mets) * 0.02
        
        enzyme_activity = gene_expr @ self.G
        enzyme_activity = F.softmax(enzyme_activity, dim=-1)
        
        gene_traj = [gene_expr.clone()]
        met_traj = [met_conc.clone()]
        
        for _ in range(n_steps):
            substrate = met_conc.mean(dim=-1, keepdim=True)
            flux = enzyme_activity * substrate / (0.5 + substrate)
            dM = flux @ self.S.T + trends + 0.001 * torch.randn_like(met_conc)
            
            met_conc = (met_conc + dt * dM).clamp(min=0.001)
            gene_expr = (gene_expr + 0.001 * torch.randn_like(gene_expr)).clamp(0.1, 3.0)
            
            gene_traj.append(gene_expr.clone())
            met_traj.append(met_conc.clone())
        
        return {
            'gene_trajectory': torch.stack(gene_traj, dim=1),
            'met_trajectory': torch.stack(met_traj, dim=1),
        }

generator = DynamicTrajectoryGenerator(S, G, n_genes, n_mets, n_rxns)

# Test
test = generator.simulate(n_steps=20, batch_size=2)
delta = (test['met_trajectory'][:, -1] - test['met_trajectory'][:, 0]).abs()
print(f"Average change: {delta.mean():.4f}")
print("✅ Dynamics are NOT flat!" if delta.mean() > 0.01 else "⚠️ Too flat")

In [ ]:
#@title 6️⃣ Train with Dynamic Loss!

# === SETTINGS ===
n_epochs = 500
batch_size = 64
n_steps = 50
lr = 3e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
loss_fn = HamiltonianDynamicsLoss(delta_weight=10.0, direction_weight=5.0)

def train_step(batch):
    gene_traj = batch['gene_trajectory'].to(device)
    met_traj = batch['met_trajectory'].to(device)
    
    B, T, _ = gene_traj.shape
    total_loss = 0
    all_metrics = {'state': 0, 'delta': 0, 'direction': 0, 'energy': 0}
    
    for t in range(T - 1):
        out = model(gene_traj[:, t], met_traj[:, t])
        pred_next = out['next_metabolites']
        
        loss, metrics = loss_fn(
            pred_next,
            met_traj[:, t + 1],
            met_traj[:, t],
            out['energies'],
        )
        total_loss += loss
        
        for k in all_metrics:
            all_metrics[k] += metrics[k]
    
    n = T - 1
    for k in all_metrics:
        all_metrics[k] /= n
    
    return total_loss / n, all_metrics

# Training
losses = []
print(f"Training: {n_epochs} epochs, batch={batch_size}, lr={lr}")
print(f"Loss weights: delta={loss_fn.delta_weight}x, direction={loss_fn.direction_weight}x")
print()

for epoch in tqdm(range(n_epochs)):
    batch = generator.simulate(n_steps=n_steps, batch_size=batch_size)
    
    optimizer.zero_grad()
    loss, metrics = train_step(batch)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:4d}: Loss={loss.item():.4f} | "
              f"state={metrics['state']:.4f} delta={metrics['delta']:.4f} "
              f"dir={metrics['direction']:.4f}")

print(f"\nFinal loss: {losses[-1]:.6f}")

In [ ]:
#@title 7️⃣ Evaluate: Check if dynamics are learned
import matplotlib.pyplot as plt

# Generate test
test = generator.simulate(n_steps=50, batch_size=1)
gene_test = test['gene_trajectory'].to(device)
met_test = test['met_trajectory'].to(device)

# Predict
# Set to eval mode
model.eval()

# Predict (no torch.no_grad - Hamiltonian needs gradients internally)
pred_traj = [met_test[:, 0]]
current = met_test[:, 0]

for t in range(50):
    out = model(gene_test[:, t], current)
    current = out['next_metabolites'].detach()
    pred_traj.append(current)

pred_traj = torch.stack(pred_traj, dim=1)

true = met_test[0].cpu().numpy()
pred = pred_traj[0].cpu().numpy()

# Metrics
corr = np.corrcoef(true.flatten(), pred.flatten())[0, 1]
mse = np.mean((true - pred)**2)

# Delta check (THE KEY)
true_delta = true[1:] - true[:-1]
pred_delta = pred[1:] - pred[:-1]
delta_corr = np.corrcoef(true_delta.flatten(), pred_delta.flatten())[0, 1]

print(f"State Correlation: {corr:.4f}")
print(f"Delta Correlation: {delta_corr:.4f}  ← THIS IS THE KEY METRIC")
print(f"MSE: {mse:.6f}")

# Plot
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for i, ax in enumerate(axes.flatten()):
    ax.plot(true[:, i], 'b-', label='True', alpha=0.7)
    ax.plot(pred[:, i], 'r--', label='Predicted', alpha=0.7)
    ax.set_title(f'Metabolite {i}')
    ax.legend()

plt.suptitle(f'Delta Correlation: {delta_corr:.3f}')
plt.tight_layout()
plt.savefig('trajectory_v3.png', dpi=150)
plt.show()

In [ ]:
#@title 8️⃣ Save Model

save_dict = {
    'model_state_dict': model.state_dict(),
    'genes': genes,
    'metabolites': metabolites,
    'stoichiometry': S,
    'gene_reaction_map': G,
    'training_losses': losses,
    'version': 'v3_hamiltonian_dynamics',
    'loss_config': {
        'delta_weight': loss_fn.delta_weight,
        'direction_weight': loss_fn.direction_weight,
    },
}

torch.save(save_dict, 'dark_manifold_v3_dynamics.pt')
print("Saved: dark_manifold_v3_dynamics.pt")

from google.colab import files
files.download('dark_manifold_v3_dynamics.pt')
files.download('trajectory_v3.png')

## Key Metrics to Watch

| Metric | v2 (old) | v3 (new) | What it means |
|--------|----------|----------|---------------|
| State Corr | 0.95+ | 0.95+ | How well absolute values match |
| **Delta Corr** | ~0.0 | **>0.5** | How well CHANGES match |
| MSE | Low | Low | Overall error |

**Delta Correlation is THE key metric.** If it's near 0, the model is predicting flat. If it's >0.5, the model learned real dynamics.